# 02 — CNN Model

This notebook builds and trains a custom 4-block Convolutional Neural Network for binary fresh/rotten classification. No pre-trained weights are used.

In [4]:
import os, time, warnings
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (Conv2D, BatchNormalization, MaxPooling2D,
                                      Flatten, Dense, Dropout, Input)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.preprocessing.image import ImageDataGenerator

ROOT       = r"D:\Hams Khaled\CS Year 3 Sem 2\Image Processing\Final Project\project"
SPLIT_DIR  = os.path.join(ROOT, "data",    "split")
MODELS_DIR = os.path.join(ROOT, "models")
PLOTS_DIR  = os.path.join(ROOT, "results", "plots")
MODEL_PATH = os.path.join(MODELS_DIR, "CNN_model.h5")

os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(PLOTS_DIR,  exist_ok=True)

IMG_SIZE   = (224, 224)
BATCH_SIZE = 32
EPOCHS     = 30
LR         = 0.001
RANDOM_STATE = 42
tf.random.set_seed(RANDOM_STATE)

print("Configuration complete.")


Configuration complete.


## Step 1 — Rebuild Data Generators

We re-create the same generators from notebook 01 so this notebook is self-contained.

In [3]:
train_datagen = ImageDataGenerator(
    rescale=1./255, horizontal_flip=True,
    rotation_range=20, zoom_range=0.2, brightness_range=[0.8, 1.2]
)
val_datagen = ImageDataGenerator(rescale=1./255)

train_gen = train_datagen.flow_from_directory(
    os.path.join(SPLIT_DIR, "train"),
    target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode="binary", shuffle=True, seed=RANDOM_STATE
)
val_gen = val_datagen.flow_from_directory(
    os.path.join(SPLIT_DIR, "val"),
    target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode="binary", shuffle=False
)
print(f"  Train samples : {train_gen.samples}")
print(f"  Val   samples : {val_gen.samples}")


Found 9518 images belonging to 2 classes.
Found 2040 images belonging to 2 classes.
  Train samples : 9518
  Val   samples : 2040


## Step 2 — Define the CNN Architecture

Four Conv→BN→MaxPool blocks progressively doubling filters (32→64→128→256), followed by a Dense head with Dropout for regularisation.

In [ ]:
model = Sequential([
    Input(shape=(224, 224, 3)),

    # Block 1
    Conv2D(32, (3, 3), activation='relu', padding='same'),
    BatchNormalization(),
    MaxPooling2D((2, 2)),

    # Block 2
    Conv2D(64, (3, 3), activation='relu', padding='same'),
    BatchNormalization(),
    MaxPooling2D((2, 2)),

    # Block 3
    Conv2D(128, (3, 3), activation='relu', padding='same'),
    BatchNormalization(),
    MaxPooling2D((2, 2)),

    # Block 4
    Conv2D(256, (3, 3), activation='relu', padding='same'),
    BatchNormalization(),
    MaxPooling2D((2, 2)),

    # Classifier head
    Flatten(),
    Dense(256, activation='relu'),
    Dropout(0.5),
    Dense(1, activation='sigmoid'),
], name="FreshRottenCNN")

model.compile(
    optimizer=Adam(learning_rate=LR),
    loss='binary_crossentropy',
    metrics=['accuracy']
)
model.summary()


## Step 3 — Train the Model

`EarlyStopping` monitors val_loss (patience=5) and `ModelCheckpoint` saves the best weights.

In [ ]:
callbacks = [
    EarlyStopping(
        monitor='val_loss', patience=5,
        restore_best_weights=True, verbose=1
    ),
    ModelCheckpoint(
        filepath=MODEL_PATH, monitor='val_loss',
        save_best_only=True, verbose=1
    ),
]

print(f"  Training for up to {EPOCHS} epochs …")
t_start = time.time()

history = model.fit(
    train_gen,
    epochs=EPOCHS,
    validation_data=val_gen,
    callbacks=callbacks,
    verbose=1,
)

t_end = time.time()
training_time = t_end - t_start
print(f"\nTraining complete in {training_time:.1f} seconds ({training_time/60:.1f} min)")
print(f"Best model saved → {MODEL_PATH}")


## Step 4 — Plot Training Curves

In [5]:
def save_curve(history_dict, metric, out_path, title):
    train_vals = history_dict[metric]
    val_vals   = history_dict[f"val_{metric}"]
    epochs_ran = range(1, len(train_vals) + 1)
    plt.figure(figsize=(8, 5))
    plt.plot(epochs_ran, train_vals, 'b-o', label=f'Train {metric.capitalize()}')
    plt.plot(epochs_ran, val_vals,   'r-o', label=f'Val {metric.capitalize()}')
    plt.title(title)
    plt.xlabel('Epoch')
    plt.ylabel(metric.capitalize())
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(out_path, dpi=150)
    plt.close()
    print(f"Saved → {out_path}")

save_curve(history.history, "accuracy",
           os.path.join(PLOTS_DIR, "CNN_accuracy.png"),
           "CNN — Training vs Validation Accuracy")

save_curve(history.history, "loss",
           os.path.join(PLOTS_DIR, "CNN_loss.png"),
           "CNN — Training vs Validation Loss")


NameError: name 'history' is not defined

## Step 5 — Summary

In [ ]:
trainable_params = sum(tf.size(w).numpy() for w in model.trainable_weights)
print("=" * 50)
print("  CNN MODEL SUMMARY")
print("=" * 50)
print(f"  Epochs run        : {len(history.history['accuracy'])}")
print(f"  Trainable params  : {trainable_params:,}")
print(f"  Training time     : {training_time:.1f} s ({training_time/60:.1f} min)")
best_val_acc  = max(history.history['val_accuracy'])
best_val_loss = min(history.history['val_loss'])
print(f"  Best val accuracy : {best_val_acc:.4f}")
print(f"  Best val loss     : {best_val_loss:.4f}")
print("Model saved to  :", MODEL_PATH)

# Persist training time for evaluation notebook
import json
meta = {"CNN_training_time": training_time, "CNN_epochs": len(history.history['accuracy']),
        "CNN_params": int(trainable_params)}
with open(os.path.join(ROOT, "results", "CNN_meta.json"), "w") as f:
    json.dump(meta, f)
print("Metadata saved.")
